<a href="https://colab.research.google.com/github/BingBingNancy/IMDA_Captchas_recognition/blob/main/test_of_IMDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install tensorflow

In [81]:
import glob
import os
import re
import numpy as np

#  Define the pattern to match all your files
# This creates a list of all files ending with '.txt'
file_list = glob.glob('*.txt')

# Print the list to confirm it found all 10 files
print(f"Found {len(file_list)} files to process: {file_list}")


input_file_list = sorted(
    [s for s in file_list if s.startswith('input')],
    key=lambda s: s[5:]
)
output_file_list = sorted(
    [s for s in file_list if s.startswith('output')],
    key=lambda s: s[6:]
)
print(input_file_list)
print(output_file_list)

#define a list X_list to store feature_matrix_per_file
X_list = []

for filename in input_file_list:
    print(f"\n--- Processing file: {filename} ---")

    #  Open the file for reading ('r')
    with open(filename, 'r') as file:
        # Read the content
        content = file.read()
    lst_content = content.split('\n')
    #print(lst_content)
    lst_content.pop(0)
    lst_content.pop(-1)
    #print(lst_content)
    #print(len(lst_content))
    numpy_content = []
    for i in range(len(lst_content)):
        ith_str = re.split(r'[,\s]+', lst_content[i])
        numpy_content.append(ith_str)
    #print(numpy_content[:3])

    RGB_matrix = [[int(s) for s in inner_list] for inner_list in numpy_content]
    #print(RGB_matrix[0])
    lenth = [len(inner_list) for inner_list in RGB_matrix]
    #print(lenth)


    #convert RGB into pixel matrix 30x60
    pixel_matrix = [[sum(row[i:i+3])/3 for i in range(0, len(row), 3)] for row in RGB_matrix]
    pixel_matrix_shape = [len(inner_list) for inner_list in pixel_matrix]
    #print(pixel_matrix_shape)
    #print(pixel_matrix[0])


    #split pixel matrix 30x60 into 5 pixel matrix (pixel1_matrix, pixel2_matrix,..pixel5_matrix). Each pixel_matrix corresponds to the feature of a letter

    pixel_matrix = np.array(pixel_matrix)
    #print(pixel_matrix.shape)

    pixel1_matrix = pixel_matrix[::,0:12]#30x12
    #print(pixel1_matrix)
    pixel2_matrix = pixel_matrix[::,12:24]#30x12
    pixel3_matrix = pixel_matrix[::,24:36]
    pixel4_matrix = pixel_matrix[::,36:48]
    pixel5_matrix = pixel_matrix[::,48:60]

    #convert pixel matrix to feature vector
    pixel1_vector = pixel1_matrix.reshape(-1)
    pixel2_vector = pixel2_matrix.reshape(-1)
    pixel3_vector = pixel3_matrix.reshape(-1)
    pixel4_vector = pixel4_matrix.reshape(-1)
    pixel5_vector = pixel5_matrix.reshape(-1)

    #print(pixel1_vector)
    #print(len(pixel1_vector)) #360

    feature_matrix_per_file = np.array([pixel1_vector, pixel2_vector, pixel3_vector, pixel4_vector, pixel5_vector])
    print(feature_matrix_per_file.shape)# 5x360. each row corresponds to the feature vector of a character.

    X_list.append(feature_matrix_per_file)


#print(len(X_list))
X =np.concatenate(X_list, axis=0)
print(X.shape)#125x360

#feature normalization
X_normalized = X.astype(np.float32) / 255.0
print(X_normalized[0])


Found 50 files to process: ['input12.txt', 'input00.txt', 'output10.txt', 'input08.txt', 'input24.txt', 'output15.txt', 'output12.txt', 'output02.txt', 'input22.txt', 'input23.txt', 'output24.txt', 'output05.txt', 'output19.txt', 'input04.txt', 'input13.txt', 'output09.txt', 'input07.txt', 'output00.txt', 'input05.txt', 'output14.txt', 'input15.txt', 'output01.txt', 'input16.txt', 'input19.txt', 'output18.txt', 'output06.txt', 'output20.txt', 'input10.txt', 'input11.txt', 'output11.txt', 'input18.txt', 'input06.txt', 'input02.txt', 'output17.txt', 'output07.txt', 'input01.txt', 'input21.txt', 'output13.txt', 'input17.txt', 'input09.txt', 'output04.txt', 'output23.txt', 'output22.txt', 'input20.txt', 'output16.txt', 'output03.txt', 'input14.txt', 'output21.txt', 'output08.txt', 'input03.txt']
['input00.txt', 'input01.txt', 'input02.txt', 'input03.txt', 'input04.txt', 'input05.txt', 'input06.txt', 'input07.txt', 'input08.txt', 'input09.txt', 'input10.txt', 'input11.txt', 'input12.txt', '

In [82]:
#encode y label
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader



y_list = []
for filename in output_file_list:
    print(f"\n--- Processing file: {filename} ---")

    # 3. Open the file for reading ('r')
    with open(filename, 'r') as file:
        # Read the content
        output_content = file.read()

    #print(len(output_content))
    output = output_content[:5]
    #print(output)
    output_list = list(output)
    #print(output_list)
    y_list.extend(output_list)

print(y_list)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_list)

print(f"Original Labels (First 10): {y_list[:10]}")
print(f"Encoded Integers (First 10): {y_encoded[:10]}")
print(f"Shape of Encoded Data: {y_encoded.shape}")


--- Processing file: output00.txt ---

--- Processing file: output01.txt ---

--- Processing file: output02.txt ---

--- Processing file: output03.txt ---

--- Processing file: output04.txt ---

--- Processing file: output05.txt ---

--- Processing file: output06.txt ---

--- Processing file: output07.txt ---

--- Processing file: output08.txt ---

--- Processing file: output09.txt ---

--- Processing file: output10.txt ---

--- Processing file: output11.txt ---

--- Processing file: output12.txt ---

--- Processing file: output13.txt ---

--- Processing file: output14.txt ---

--- Processing file: output15.txt ---

--- Processing file: output16.txt ---

--- Processing file: output17.txt ---

--- Processing file: output18.txt ---

--- Processing file: output19.txt ---

--- Processing file: output20.txt ---

--- Processing file: output21.txt ---

--- Processing file: output22.txt ---

--- Processing file: output23.txt ---

--- Processing file: output24.txt ---
['E', 'G', 'Y', 'K', '4',

In [83]:
# Split 80% for training and 20% for testing (100 samples train, 25 samples test)



X_train, X_test, y_train, y_test = train_test_split(
    X_normalized,
    y_encoded,
    test_size=0.2,
    random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (100, 360)
y_train shape: (100,)
X_test shape: (25, 360)
y_test shape: (25,)


In [84]:
#dataloader
class CharDataset(Dataset):
    def __init__(self, X_data, y_data):
        # Convert X to float tensor, y to LongTensor (required by CrossEntropyLoss)
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.long) # Use torch.long for indices

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create DataLoader instances
train_loader = DataLoader(CharDataset(X_train, y_train), batch_size=5, shuffle=True)
test_loader = DataLoader(CharDataset(X_test, y_test), batch_size=5, shuffle=False)

# Get parameters
INPUT_SIZE = X_train.shape[1]  # 36
OUTPUT_SIZE = len(label_encoder.classes_) # 36
print(OUTPUT_SIZE)

36


In [116]:
# Define the Multi-Class MLP Model ---
# class MultiClassMLP(nn.Module):
#     def __init__(self, input_size, output_size):
#         super(MultiClassMLP, self).__init__()
#         # 36 -> 64
#         self.layer1 = nn.Linear(input_size, 64)
#         # 64 -> 36
#         self.output_layer = nn.Linear(64, output_size)
#         self.dropout = nn.Dropout(0.6) # Increase from 0.5 to 0.6 or 0.7

#     def forward(self, x):
#         x = torch.relu(self.layer1(x))
#         # Softmax ensures outputs sum to 1, providing class probabilities

#         x = torch.softmax(self.output_layer(x), dim=1)
#         return x

class MultiClassMLP(nn.Module):
    def __init__(self, input_size, output_size):
        super(MultiClassMLP, self).__init__()
        # 360 features -> 64 neurons
        self.layer1 = nn.Linear(input_size, 64)
        # 64 neurons -> 36 output classes
        self.output_layer = nn.Linear(64, output_size)
        self.dropout = nn.Dropout(0.6)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.dropout(x)

        # --- CRITICAL FIX HERE ---
        # Return the raw output (logits) for nn.CrossEntropyLoss.
        # DO NOT apply torch.softmax here.
        x = self.output_layer(x)
        return x


model = MultiClassMLP(INPUT_SIZE, OUTPUT_SIZE)

#  Initialize Loss and Optimizer
# nn.CrossEntropyLoss is required for Multi-Class classification with integer labels
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001,  weight_decay=0.001)
#optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

#  Simplified Training Loop ---
NUM_EPOCHS = 100

for epoch in range(NUM_EPOCHS):
    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)

        # Loss calculation uses the raw output scores from the model
        # The labels must be long/int indices (0 to 35)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
    if (epoch+1)%20==0:
      print(f'Epoch [{epoch + 1}/{NUM_EPOCHS}], Loss: {loss.item():.3f}')

Epoch [20/100], Loss: 3.293
Epoch [40/100], Loss: 2.914
Epoch [60/100], Loss: 3.023
Epoch [80/100], Loss: 2.721
Epoch [100/100], Loss: 2.381


In [94]:
# Create 5 new samples for demonstration
X_new_raw =  X_normalized[:5] # suppose we use the first image input00
X_new_tensor = torch.tensor(X_new_raw, dtype=torch.float32)

model.eval() # Set model to evaluation mode
with torch.no_grad():
    # Model outputs 36 probability scores for each sample
    probability_scores = model(X_new_tensor)

#  Convert Probabilities to Character Labels
# Find the index of the highest probability (the predicted class)
predicted_indices = torch.argmax(probability_scores, dim=1).numpy()

# Use the LabelEncoder to convert indices (0-35) back to character labels ('A', 'B', etc.)
predicted_characters = label_encoder.inverse_transform(predicted_indices)

print("-" * 50)
print("Classification of New Data (5 Samples):")
print("-" * 50)

result = []
for i in range(len(X_new_raw)):
    # Get the probability confidence for the predicted class
    confidence = probability_scores[i, predicted_indices[i]].item()
    result.append(predicted_characters[i])
    print(f"{i+1}th character in the image:")
    print(f"  Predicted Label: {predicted_characters[i]}")
    #print(f"  Confidence: {confidence*100:.2f}%")



final_result = "".join(result)
print('The captchas is :',final_result)


--------------------------------------------------
Classification of New Data (5 Samples):
--------------------------------------------------
1th character in the image:
  Predicted Label: V
2th character in the image:
  Predicted Label: V
3th character in the image:
  Predicted Label: V
4th character in the image:
  Predicted Label: V
5th character in the image:
  Predicted Label: V
The captchas is : VVVVV


In [ ]:
import re
lst_content = content.split('\n')
print(lst_content)
lst_content.pop(0)
lst_content.pop(-1)
print(lst_content)
print(len(lst_content))
numpy_content = []
for i in range(len(lst_content)):
    ith_str = re.split(r'[,\s]+', lst_content[i])
    numpy_content.append(ith_str)
print(numpy_content[:3])

RGB_matrix = [[int(s) for s in inner_list] for inner_list in numpy_content]
print(RGB_matrix[0])
lenth = [len(inner_list) for inner_list in RGB_matrix]
print(lenth)


['30 60', '255,255,255 250,250,250 254,254,254 250,250,250 254,254,254 255,255,255 253,253,253 254,254,254 255,255,255 255,255,255 255,255,255 250,250,250 255,255,255 249,249,249 255,255,255 253,253,253 254,254,254 233,233,233 255,255,255 249,249,249 255,255,255 239,239,239 195,195,195 247,247,247 245,245,245 249,249,249 252,252,252 247,247,247 254,254,254 255,255,255 246,246,246 254,254,254 255,255,255 178,178,178 203,203,203 192,192,192 187,187,187 195,195,195 190,190,190 189,189,189 193,193,193 196,196,196 188,188,188 186,186,186 201,201,201 180,180,180 255,255,255 255,255,255 240,240,240 255,255,255 250,250,250 255,255,255 255,255,255 247,247,247 252,252,252 248,248,248 245,245,245 255,255,255 242,242,242 251,251,251', '255,255,255 253,253,253 255,255,255 255,255,255 255,255,255 255,255,255 254,254,254 255,255,255 248,248,248 255,255,255 250,250,250 253,253,253 250,250,250 251,251,251 250,250,250 250,250,250 255,255,255 255,255,255 242,242,242 249,249,249 255,255,255 255,255,255 18

In [ ]:
#convert RGB into pixel matrix 30x60
pixel_matrix = [[sum(row[i:i+3])/3 for i in range(0, len(row), 3)] for row in RGB_matrix]
pixel_matrix_shape = [len(inner_list) for inner_list in pixel_matrix]
print(pixel_matrix_shape)
print(pixel_matrix[0])

[60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60]
[255.0, 250.0, 254.0, 250.0, 254.0, 255.0, 253.0, 254.0, 255.0, 255.0, 255.0, 250.0, 255.0, 249.0, 255.0, 253.0, 254.0, 233.0, 255.0, 249.0, 255.0, 239.0, 195.0, 247.0, 245.0, 249.0, 252.0, 247.0, 254.0, 255.0, 246.0, 254.0, 255.0, 178.0, 203.0, 192.0, 187.0, 195.0, 190.0, 189.0, 193.0, 196.0, 188.0, 186.0, 201.0, 180.0, 255.0, 255.0, 240.0, 255.0, 250.0, 255.0, 255.0, 247.0, 252.0, 248.0, 245.0, 255.0, 242.0, 251.0]


In [ ]:
#split pixel matrix 30x60 into 5 pixel matrix (pixel1_matrix, pixel2_matrix,..pixel5_matrix). Each pixel_matrix corresponds to the feature of a letter
import numpy as np
pixel_matrix = np.array(pixel_matrix)
print(pixel_matrix.shape)

pixel1_matrix = pixel_matrix[::,0:12]#30x12
print(pixel1_matrix)
pixel2_matrix = pixel_matrix[::,12:24]#30x12
pixel3_matrix = pixel_matrix[::,24:36]
pixel4_matrix = pixel_matrix[::,36:48]
pixel5_matrix = pixel_matrix[::,48:60]

#convert pixel matrix to feature vector
pixel1_vector = pixel1_matrix.reshape(-1)
pixel2_vector = pixel2_matrix.reshape(-1)
pixel3_vector = pixel3_matrix.reshape(-1)
pixel4_vector = pixel4_matrix.reshape(-1)
pixel5_vector = pixel5_matrix.reshape(-1)

print(pixel1_vector)
print(len(pixel1_vector)) #360

feature_matrix_per_file = np.array([pixel1_vector, pixel2_vector, pixel3_vector, pixel4_vector, pixel5_vector])
print(feature_matrix_per_file.shape)# 5x360. each row corresponds to the feature vector of a character.


(30, 60)
[[255. 250. 254. 250. 254. 255. 253. 254. 255. 255. 255. 250.]
 [255. 253. 255. 255. 255. 255. 254. 255. 248. 255. 250. 253.]
 [251. 247. 251. 250. 252. 255. 240. 255. 248. 255. 253. 255.]
 [195. 191. 195. 195. 194. 198. 170. 198. 188. 195. 188. 172.]
 [253. 249. 251. 252. 247. 251. 209. 250. 255. 255. 255. 210.]
 [255. 253. 254. 255. 248. 253. 197. 251. 249. 240. 251. 188.]
 [255. 253. 253. 255. 248. 254. 189. 253. 254. 254. 255. 193.]
 [255. 252. 253. 255. 249. 255. 187. 255. 250. 255. 252. 179.]
 [255. 255. 253. 244. 246. 255. 197. 253. 254. 255. 255. 181.]
 [255. 245. 249. 255. 255. 248. 185. 241. 235. 255. 246. 200.]
 [244. 251. 255. 255. 238. 255. 171. 219. 197. 198. 174. 209.]
 [255. 255. 241. 240. 255.   0.   0.  18.   0.   3.  18.   0.]
 [247. 241. 255. 255. 243.   0.   0. 250. 248. 248. 237. 250.]
 [253. 248. 255. 239. 255.   2.  15. 250. 255. 245. 255. 255.]
 [255. 246. 255. 255. 234.   0.   0. 249. 255. 231. 255. 243.]
 [251. 255. 239. 240. 255.   0.   0.   2.   0.

In [ ]:
import numpy as np
# ... define arr1, arr2, arr3 ...
arr1 = np.array([1, 2, 3])
arr2 = np.array([4, 5, 6])
arr3 = np.array([7, 8, 9])

arr4 = np.array([arr1, arr2, arr3])
print(arr4)

[[1 2 3]
 [4 5 6]
 [7 8 9]]


In [ ]:
output_file_path = 'output00.txt'

# Open the file and read its contents
with open(output_file_path, 'r') as file:
    output_content = file.read()

print(len(output_content))
output = output_content[:5]
print(output)
output_list = list(output)




6
EGYK4
['E', 'G', 'Y', 'K', '4']
